# 解答例（教員用）　第15回
***
> このノートブックは **模範解答の一例**です．コードの空欄（`___`）を埋め，探索は複数ラウンドに分けて実行し，解答欄には実際に得られた出力の数値を根拠に書いています．
> 乱数を使う箇所は固定していますが，ライブラリのバージョンや実行環境によって数値が **小数第2〜3位ほどずれる**ことがあります．探索型の問題なので，**別の探索経路・別の最良設定でも，根拠が筋の通っていれば正解**です．

# 第15回　ニューラルネットワーク実装（全結合1層）
***
> **前提**: 第14回 PyTorch 基礎の続きです。第7回では `MyLinearRegression` クラスを自作し `fit` / `predict` を実装しました。PyTorch では `nn.Module` を継承し `forward` メソッドで同様の「入力→出力」の計算を定義します。
>
> | 第7回 `MyLinearRegression` | 第15回 `nn.Module` |
> |---|---|
> | `fit(X, y)` で重みを学習 | 学習ループ + `optimizer.step()` で重みを更新 |
> | `predict(X)` で予測 | `forward(x)` で順伝播 |
> | `score(X, y)` で R² | テストデータで正解率を計算 |

> ⚠️ **この課題で身につけること：コーディングではなく「AI（ニューラルネット）の中身の理解」です。**
>
> コードは AI に書かせても構いません。重要なのは「**なぜその処理を選ぶのか**」「**パラメータや特徴量を変えると結果がどう変わるのか**」を理解し、提出物で示すことです。各問には学習目標を示すタグが付いています。
>
> | タグ | 意味 | あなたがすること |
> |---|---|---|
> | **【骨格】** | 動くコードは与えられている | 設計上の決定点（数値・選択肢・特徴量）だけを変更する |
> | **【選択】** | 適切な手法を選ぶ問題 | 複数候補から選び、**理由**を解答用コードセルに書く |
> | **【実験】** | 試行錯誤の記録 | パラメータ等を変えて結果を表に記録し、**考察**する |
> | **【説明】** | 理解の証跡 | 与えられたコードの各行に `# 説明:` で意味を書く |
> | **【探索】** | 最適設定を自分で探す | **仮説 → 試行 → 記録 → 次の仮説** を何度も繰り返す。全試行が自動で履歴に残る |
> | **【指示書】** | 知識を LLM に渡す | 実験で得た知識を、LLM が使える「指示書」に書き起こし、LLM の出力を自分のログで採点する |
>
> コードは原則として完成形ですが、**各問の「核心となる最低限の数行」は `# ★あなたが書く★` として空欄**にしてあります。AI に頼り切らず、要となる処理は自分で書けることも確認します（ボイラープレートは提供済み）。
>
> 各問の **✍️ 解答用コードセル**（`# (1-a)` 形式の変数・文字列）に、設計判断・理由・実験結果・考察を**項目ごとに**記入してください。これが採点対象です。

---

## 🧪 この回の進め方：最適な設定は「自分で」探す

この回では **「正解のパラメータ」は教えません**。答えは、あなたが何度も試して見つけます。

```
  ①仮説を立てる → ②試す（1回1行の記録が自動で残る） → ③結果を見て仮説を直す → ①へ戻る
```

### 試行錯誤の記録帳 `Lab`（最初のコードセルで用意済み。中身を読んで使い方を理解すること）

| 使い方 | 何をするか |
|---|---|
| `lab.df()` | これまでの全試行を表で見る（試行番号・設定・スコア・仮説） |
| `lab.plot(x="lr", group=...)` | 設定とスコアの関係、ベスト更新の推移をグラフにする |
| `lab.report()` | 探索の自己診断（回数・範囲・端に張り付いていないか・仮説を書いたか） |
| `lab.final_test(...)` | **テストデータでの最終評価。1回しか実行できない** |
| `lab.save()` | 全履歴を CSV に保存する |

### ルール（ここが「AIの理解」の中心）

1. **探索に使ってよいのは検証スコア（訓練データの一部を取り分けた検証用データでの正解率）だけ**。テストデータは最後に **1回だけ**（見てから設定を変えたらデータリーク）。
2. **試行のたびに「仮説」を書く**（何を確かめたい試行か）。やみくもな総当たりは評価されません。
3. **粗く → 細かく**：まず桁・範囲を広く振って良さそうな所を掴み、次にその周辺を細かく調べる。
4. **ベストが探索範囲の端に来たら、範囲を広げて確かめる**。
5. 上位の設定同士の差が、ばらつきより小さいなら「差がある」とは言えない。
6. **実験の前に予測を書く**。予測と結果のズレが、あなたが得た「知識」の証拠になる。
7. 最後に、得た知識を **LLM への指示書**にまとめる（最終問題）。

### 評価の観点

| 観点 | 見られること |
|---|---|
| 試行の量と質 | 目安回数以上・仮説が毎回ある・粗→細の流れがある |
| 結果の読み取り | 外れた仮説から何を学んだか（試行番号 `#n` を引用して書く） |
| 検証の作法 | テストは1回だけ／検証スコアとテストの差を正しく解釈している |
| 知識の言語化 | 自分の実験結果を、LLM が実行できる指示書に落とせている |

## 目次
1. SimpleMLP の実装
2. 学習ループ
3. 損失曲線
4. テスト正解率
5. 実験で得た知識を LLM への指示書にする【指示書】

---

## この回で学ぶこと

### ニューラルネットワークの構造

多層パーセプトロン（MLP：Multilayer Perceptron）は，複数の「全結合層（Linear層）」を重ねたシンプルなニューラルネットワークだ：

```
入力層 (784次元)
   ↓ nn.Linear(784, 128) 重み行列 W₁ (784×128) + バイアス b₁
隠れ層 (128次元)
   ↓ ReLU(x) = max(0, x)  活性化関数
   ↓ nn.Linear(128, 10)  重み行列 W₂ (128×10) + バイアス b₂
出力層 (10次元) ← 数字0〜9のスコア（logits）
```

全部で 784×128 + 128 + 128×10 + 10 = **101,770個のパラメータ**が学習される。

### なぜ活性化関数（ReLU）が必要か

線形層だけを重ねると，どれだけ重ねても「1つの線形変換」と等価になってしまう（行列の積は行列）。**非線形な活性化関数**を挟むことで，複雑な非線形パターンを学習できるようになる。

ReLU（Rectified Linear Unit）: `f(x) = max(0, x)`
- 計算が単純で速い
- **勾配消失問題を回避**（sigmoid や tanh は深いネットで勾配が消えやすかった）
- 現在のデフォルト選択

### 誤差逆伝播法（Backpropagation）の流れ

```
① forward(x)  : 入力から出力を計算（順伝播）
② loss = criterion(output, label)  : 損失を計算
③ optimizer.zero_grad()  : 前のステップの勾配をリセット
④ loss.backward()  : 各パラメータの勾配を計算（逆伝播）
⑤ optimizer.step()  : 勾配を使ってパラメータを更新
```

**③ `zero_grad()` を忘れると**：勾配が前のステップと加算され，誤った更新が起きる。必ずループの最初でリセットすること。

### Adam オプティマイザとは

勾配降下法（SGD）の改良版で，以下の特徴を持つ：
- **各パラメータに個別の学習率**を自動調整
- モーメンタム（過去の勾配の方向を考慮）
- `lr=0.001` が一般的なデフォルト設定

SGD より収束が速く，初心者にも扱いやすいため，現在の標準的な選択肢となっている。

---

## 📚 将来の自分のためのメモ — 大学研究での応用


### どんな研究で使われるか

| 分野 | 具体例 |
|---|---|
| 情報科学 | 画像分類のベースライン MLP，表データの深層学習 |
| 医学 | 検査値の多次元パターンからの診断支援 |
| 工学 | センサー信号の分類，異常検知の入門モデル |
| 認知科学 | 単純な知覚課題のニューラルネットモデル（比較用） |

### 応用できる場面

- **非線形パターン**があり，線形モデルでは足りない表データ・ベクトルデータ
- PyTorch で**学習ループを自分で制御**したい（損失の可視化，カスタム損失）
- 深層学習の**最小構成**を理解してから CNN / Transformer に進みたい
- MNIST のような**整ったベンチマーク**で手法を比較する

### 応用しにくい・向かない場面

- **画像データ**— 空間構造を活かせない MLP は CNN に劣る（第16回）
- **サンプル数が極端に少ない**— 10万パラメータ級のモデルは過学習しやすい
- **解釈可能性**が最優先（係数，決定木の方が向く）
- 学習曲線が改善しないのに**層を深くするだけ**— データ量・正則化・学習率の見直しが先
- `zero_grad()` 忘れなど実装バグで「再現できない結果」になりやすい— コードレビューが重要

### 研究で報告するときのポイント

1. **訓練損失と検証損失の曲線**を図示（過学習の有無）
2. 層の幅，学習率，エポック数，バッチサイズを報告
3. 同条件で**ベースライン**（線形モデル，浅い MLP）と比較
4. テストデータは**ハイパーパラメータ調整に使わない**（第13回と同じ原則）

### 関連する発展トピック（調べてみると良い）

- Dropout，Batch Normalization（過学習・学習安定化）
- Learning rate scheduler，Early stopping
- Weight decay（L2 正則化— 第9回との対応）

In [ ]:
%pip install -q torch torchvision


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import pandas as pd
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

DATA_ROOT = "./data"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.ToTensor()
train_dataset = datasets.MNIST(root=DATA_ROOT, train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root=DATA_ROOT, train=False, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# 検証用データ：訓練データの 50000〜52000 番を取り分ける（学習に使わない）。test_loader は最終評価の1回だけに使う
val_loader = DataLoader(Subset(train_dataset, range(50000, 52000)), batch_size=500, shuffle=False)


# ==== 試行錯誤の記録帳（この回の全実験がここに溜まる。中身を読んで使い方を理解しておくこと）====
class Lab:
    """試行錯誤の記録帳。何回でも log() でき、表・グラフ・診断レポートにできる。
    ・スコア（score）は「検証用スコア（CV）」だけを記録する。テストデータは final_test() で1回だけ。"""

    def __init__(self, name, score_name="cv_score", higher_is_better=True):
        self.name, self.score_name, self.hib = name, score_name, higher_is_better
        self.rows, self.test_result = [], None

    def seen(self, params):
        return any(r["params"] == params for r in self.rows)

    def log(self, hypothesis, params, score, **extra):
        """1試行を記録する。同じ params を再度記録しようとすると無視する（重複の水増し防止）。"""
        if self.seen(params):
            print(f"  (skip) 既に試した設定です: {params}")
            return
        self.rows.append(dict(trial=len(self.rows) + 1, hypothesis=hypothesis.strip(),
                              params=dict(params), score=float(score), extra=extra))
        best = self.best()["trial"] == len(self.rows)
        print(f"  #{len(self.rows):02d} {params}  {self.score_name}={score:.4f}"
              + "".join(f"  {k}={v}" for k, v in extra.items()) + ("  ★ベスト更新" if best else ""))

    def df(self):
        import pandas as pd
        return pd.DataFrame([{"trial": r["trial"], **r["params"], self.score_name: round(r["score"], 4),
                              **r["extra"], "hypothesis": r["hypothesis"]} for r in self.rows])

    def best(self):
        return (max if self.hib else min)(self.rows, key=lambda r: r["score"])

    def plot(self, x, group=None, logx=True, query=None):
        """パラメータ x とスコアの関係（group ごとに線を分ける）と、試行回数ごとのベスト更新の推移。"""
        import matplotlib.pyplot as plt
        d = self.df()
        dd = d.query(query) if query else d
        fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
        for g, sub in (dd.groupby(group) if group else [("all", dd)]):
            sub = sub.sort_values(x)
            ax[0].plot(sub[x], sub[self.score_name], "o-", label=str(g))
        if logx and dd[x].dtype.kind in "if" and (dd[x] > 0).all(): ax[0].set_xscale("log")
        ax[0].set_xlabel(x); ax[0].set_ylabel(self.score_name); ax[0].legend(); ax[0].set_title("Parameter vs. validation score")
        cur = d[self.score_name].cummax() if self.hib else d[self.score_name].cummin()
        ax[1].step(d["trial"], cur, where="post"); ax[1].plot(d["trial"], d[self.score_name], "x", alpha=.5)
        ax[1].set_xlabel("Trial number"); ax[1].set_title("Best score so far (flat = plateau)")
        plt.tight_layout(); plt.show()

    def report(self, min_trials=15):
        """探索の質の自己診断：回数・カバー範囲・端に張り付いていないか・仮説を書いたか。"""
        import numpy as np
        d, n = self.df(), len(self.rows)
        print(f"■ 試行回数: {n}（目安 {min_trials} 回以上） → {'OK' if n >= min_trials else 'まだ足りない'}")
        keys = list(self.rows[0]["params"]) if n else []
        b = self.best()
        print(f"■ ベスト: trial#{b['trial']} {b['params']}  {self.score_name}={b['score']:.4f}")
        for k in keys:
            vals = [r["params"][k] for r in self.rows]
            uniq = sorted(set(map(str, vals)))
            line = f"■ {k}: {len(uniq)} 種類を試した"
            nums = [v for v in vals if isinstance(v, (int, float)) and not isinstance(v, bool)]
            if len(nums) == len(vals) and len(set(nums)) > 2:
                edge = k != "seed" and b["params"][k] in (min(nums), max(nums))
                line += f"（範囲 {min(nums):g}〜{max(nums):g}）" + (" ⚠ ベストが探索範囲の端 → 範囲を広げて確かめよう" if edge else "")
            print(line)
        w = sum(1 for r in self.rows if len(r["hypothesis"]) >= 8)
        print(f"■ 仮説を書いた試行: {w}/{n}" + ("" if w == n else " ⚠ 何を確かめる試行か毎回書こう"))
        if n >= 6:
            third = n // 3
            print(f"■ 前半の最高 {d[self.score_name][:third].max():.4f} → 後半の最高 {d[self.score_name][-third:].max():.4f}")

    def final_test(self, score_fn):
        """テストデータでの最終評価。1回しか実行できない（2回目以降はエラー）。"""
        if self.test_result is not None:
            raise RuntimeError(f"テストは既に使用済みです（結果 {self.test_result:.4f}）。テストを見て設定を変えるのはデータリークです。")
        self.test_result = float(score_fn())
        print(f"最終テストスコア = {self.test_result:.4f}   （検証ベスト = {self.best()['score']:.4f}）")
        return self.test_result

    def save(self, path=None):
        path = path or f"{self.name}_log.csv"
        self.df().to_csv(path, index=False, encoding="utf-8-sig"); print("保存:", path)


## 問題1　SimpleMLP の構造を読み解く　【説明】
***

### `nn.Module` の仕組み

PyTorch のモデルは `nn.Module` を継承したクラスとして実装する。覚えるべきルールは2つ：

1. `__init__` で使用する層を**定義**する（ここでパラメータが初期化される）
2. `forward` で入力から出力への**計算を記述**する

```python
class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()  # ← 必須！親クラスの初期化
        # 層を定義
        self.fc1 = nn.Linear(784, 128)  # 重み行列 (784, 128) を作成
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        # 入力 x: (batch_size, 1, 28, 28)
        x = x.view(-1, 784)       # → (batch_size, 784) に flatten
        x = self.relu(self.fc1(x))  # → 線形変換 → ReLU → (batch_size, 128)
        return self.fc2(x)        # → (batch_size, 10) logits を返す
```

### 出力層に softmax/sigmoid がない理由

`CrossEntropyLoss`（問題2で使用）は内部で softmax を計算するため，`forward` の出力は「確率」ではなく「生スコア（logits）」でよい。推論時（`predict_proba`に相当）には `torch.softmax(output, dim=1)` を別途適用する。

### 課題

下のコードセルには `SimpleMLP`（全結合1層の MLP）が **完成形で実装**されています。今回はネットワークを一から書くのではなく、**「なぜこの構造で数字が分類できるのか」を読み解く**のが目的です。

**各行の `# 説明:` の右に、その行が何をしているかを自分の言葉で書いて**ください（コードは変更しない）。書き終えたらセルを実行し、`print(model)` の出力で構造を確認してください。

説明を書くときは、次の問いを意識してください：

- `nn.Linear(28*28, 128)` の `784` と `128` はそれぞれ何の数を表しているか？
- なぜ層と層の間に `ReLU`（非線形）を挟む必要があるのか？ 線形層だけを重ねるとどうなるか？
- `forward` の `x.view(x.size(0), -1)`（flatten）は画像をどんな形に変えているか？
- 出力層が `10` で、しかも softmax を付けていないのはなぜか？（ヒント：問題2の `CrossEntropyLoss`）

### 入力の中身を見てみよう（実行するだけ）

このあと作るネットワークに入る 1 バッチ分のデータです．**形（shape）**を確認してください．

In [2]:
# === データの中身を確認する（実行するだけ。コードは変更しない）===
_images, _labels = next(iter(train_loader))
print("画像バッチの形:", _images.shape, "（バッチ, チャンネル, 高さ, 幅）  dtype:", _images.dtype)
print("ラベルの形   :", _labels.shape, " 先頭 10 個:", _labels[:10].tolist())
print("画素値の範囲 :", _images.min().item(), "〜", _images.max().item())

画像バッチの形: torch.Size([64, 1, 28, 28]) （バッチ, チャンネル, 高さ, 幅）  dtype: torch.float32
ラベルの形   : torch.Size([64])  先頭 10 個: [6, 2, 1, 9, 5, 8, 3, 1, 3, 3]
画素値の範囲 : 0.0 〜 1.0


In [3]:
# 完成形の SimpleMLP です。各行の「# 説明:」に自分の言葉で意味を書いてください。
# （コードは変更しない。説明は AI に書かせず自分で書くこと）

class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()                       # 説明: 親クラス nn.Module の初期化を呼ぶ．これをしないとパラメータの登録などの nn.Module の機能が使えない．
        self.fc1 = nn.Linear(28 * 28, 128)       # 説明: 全結合層（入力 784 個 → 出力 128 個）．784 は 28×28 画素を並べた入力の次元数，128 は隠れ層のユニット数（特徴の数）．
        self.relu = nn.ReLU()                    # 説明: 活性化関数 ReLU（負の値を 0 にする）．線形層だけを重ねても全体が 1 つの線形変換にしかならないので，間に非線形を挟んで複雑な関数を表現できるようにする．
        self.fc2 = nn.Linear(128, 10)            # 説明: 全結合層（128 → 10）．出力は 0〜9 の 10 クラスに対応する 10 個のスコア．

    def forward(self, x):
        x = x.view(x.size(0), -1)                # 説明: (バッチ, 1, 28, 28) の画像を (バッチ, 784) に平坦化（flatten）する．x.size(0) はバッチサイズを保ち，-1 は残りの次元を自動で 784 にする．
        x = self.relu(self.fc1(x))               # 説明: 全結合層で 784 → 128 に線形変換し，ReLU で非線形にする．
        return self.fc2(x)                       # 説明: 128 → 10 の線形変換の出力（logits＝未正規化のスコア）をそのまま返す．softmax は CrossEntropyLoss の内部で行われる．


model = SimpleMLP().to(device)
print(model)


SimpleMLP(
  (fc1): Linear(in_features=784, out_features=128, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)


In [4]:
# === ✍️ 問題1 解答（採点対象）===
# 主な提出物は上のコードセルへの # 説明: 記入。以下も記入すること。

# (1-a) `nn.Linear(28*28, 128)` の `784` と `128` はそれぞれ何を表すか：
answer_1_a = """
784 は入力の次元数（28×28 = 784 画素を 1 列に並べたもの）、128 は隠れ層のユニット数（この層が作る特徴の数）。つまり 784 個の入力から 128 個の特徴を計算する全結合層。
"""

# (1-b) なぜ層と層の間に ReLU（非線形）が必要か（線形層だけを重ねるとどうなるか）
answer_1_b = """
線形層だけを何層重ねても、合成すると 1 つの線形変換（行列の積）にしかならず、直線的な境界しか作れない。ReLU のような非線形関数を間に挟むと、層を重ねるごとに複雑な（曲がった）境界や特徴を表現できるようになる。
"""

# (1-c) `forward` の `x.view(x.size(0), -1)`（flatten）は画像をどんな形に変えるか
answer_1_c = """
(バッチ, 1, 28, 28) の画像を (バッチ, 784) の 2 次元（各行が 1 枚の画像を 1 列に並べたもの）に変える。x.size(0) でバッチサイズを保ち、-1 で残りの次元（1×28×28）を自動計算する。
"""

# (1-d) 出力層が 10 で、softmax を付けていない理由
answer_1_d = """
10 は 0〜9 の 10 クラスに対応する出力（各クラスのスコア）。softmax を付けないのは、損失関数 CrossEntropyLoss が内部で softmax（＋対数）を計算するので、モデルは logits（生のスコア）を返せばよく、二重に適用すると数値的にも不安定になるから。
"""



## 問題2　学習ループを読み解く　【説明】
***

### 学習ループの各ステップ解説

```python
for epoch in range(3):          # ← 全データを3周する
    model.train()               # ← 訓練モードに切り替え（Dropout等が有効になる）
    for images, labels in train_loader:   # ← ミニバッチを順に取り出す
        images = images.to(device)        # ← データをGPU/CPUに送る
        labels = labels.to(device)

        optimizer.zero_grad()             # ← ① 勾配をリセット（必須！）
        outputs = model(images)           # ← ② forward: 予測を計算
        loss = criterion(outputs, labels) # ← ③ 損失を計算
        loss.backward()                   # ← ④ backward: 勾配を計算
        optimizer.step()                  # ← ⑤ パラメータを更新
```

### CrossEntropyLoss の意味

多クラス分類の標準的な損失関数だ：
```
CrossEntropyLoss = -Σ y_true × log(softmax(y_pred))
```
- 正解クラスの予測確率が高いほど損失が小さい（0に近づく）
- 損失が下がり続ければ，学習が正しく進んでいることを示す

### `loss.item()` の必要性

`loss` は Tensor であり，計算グラフを保持している。`.item()` を使うと Python の float に変換され，メモリが解放される。ループ内で `.item()` せずに損失を蓄積すると，計算グラフが蓄積されてメモリがあふれる危険がある。

### 課題

下のコードセルには、`SimpleMLP` を MNIST で学習する **完成形の学習ループ**があります（epoch=3, `CrossEntropyLoss`, `Adam(lr=0.001)`）。学習ループは「ニューラルネットがどうやって賢くなるのか」の中心です。

**各行の `# 説明:` の右に、その行が何をしているかを自分の言葉で書いて**ください（コードは変更しない）。書き終えたら実行し、**平均訓練損失がエポックごとに下がる**ことを確認してください。

特に次の3行（誤差逆伝播の中心）は丁寧に説明してください：

- `optimizer.zero_grad()` … なぜ毎ループの先頭で勾配をリセットするのか？ 忘れると何が起きるか？
- `loss.backward()` … ここで何が計算されるのか？（順伝播 forward との違い）
- `optimizer.step()` … `backward()` で求めた何を使って、何を更新するのか？

> **補足**: `total_loss += loss.item()` の `.item()` は何のために必要でしょうか？（ヒント：`loss` は計算グラフを保持した Tensor）

In [5]:
# 完成形の学習ループです。各行の「# 説明:」に自分の言葉で意味を書いてください。
# （コードは変更しない。説明は自分で書くこと）

criterion = nn.CrossEntropyLoss()                          # 説明: 多クラス分類の損失関数．softmax と負の対数尤度を内部でまとめて計算するので，モデルの出力（logits）をそのまま渡せる．
optimizer = optim.Adam(model.parameters(), lr=0.001)       # 説明: 最適化手法 Adam を作る．model.parameters()（学習で更新するすべての重みとバイアス）を更新対象とし，lr=0.001 は 1 ステップで重みを動かす大きさ（学習率）．

EPOCHS = 3
for epoch in range(EPOCHS):
    model.train()                                          # 説明: モデルを訓練モードにする（Dropout や BatchNorm など学習時と推論時で挙動が変わる層を学習時の動作にする）．
    total_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)  # 説明: 画像とラベルを計算するデバイス（CPU/GPU）に転送する．モデルと同じデバイスに置かないとエラーになる．
        optimizer.zero_grad()                              # 説明: 前のバッチで計算した勾配をゼロにリセットする．PyTorch は勾配を加算していくので，忘れると前のバッチの勾配が混ざって間違った方向に更新される．
        outputs = model(images)                            # 説明: 順伝播（forward）．画像をモデルに通して 10 クラスのスコア（logits）を得る．
        loss = criterion(outputs, labels)                  # 説明: 予測（outputs）と正解（labels）のずれを損失（1 つのスカラー）として計算する．
        loss.backward()                                    # 説明: 逆伝播．損失に対する各パラメータの勾配（どちらに動かすと損失が減るか）を，連鎖律で自動計算して各パラメータの .grad に入れる．
        optimizer.step()                                   # 説明: backward で求めた勾配を使って，Adam の規則で全パラメータを 1 ステップ更新する．
        total_loss += loss.item()                          # 説明: loss は計算グラフを持つ Tensor なので，.item() で Python の数値だけを取り出して足す．そのまま足すとグラフが保持されてメモリを消費し続ける．
    print(f"epoch {epoch + 1}: 平均訓練損失 = {total_loss / len(train_loader):.4f}")


epoch 1: 平均訓練損失 = 0.3463
epoch 2: 平均訓練損失 = 0.1597
epoch 3: 平均訓練損失 = 0.1124


In [6]:
# === ✍️ 問題2 解答（採点対象）===
# 主な提出物は上のコードセルへの # 説明: 記入。以下も記入すること。

# (2-a) `optimizer.zero_grad()` を忘れると何が起きるか
answer_2_a = """
PyTorch は勾配を .grad に「加算」していくので、リセットしないと前のバッチの勾配が残ったまま今のバッチの勾配が足され、間違った（巨大な）勾配でパラメータが更新されて学習がおかしくなる。
"""

# (2-b) `loss.backward()` では何が計算されるか（順伝播 forward との違い）
answer_2_b = """
loss.backward() は、損失に対する各パラメータの勾配（そのパラメータを少し動かしたとき損失がどれだけ変わるか）を、連鎖律で出力側から入力側へ逆にたどって計算し、.grad に格納する。順伝播（forward）は入力から損失の値を計算するのに対し、backward は損失を減らす方向（勾配）を計算する。
"""

# (2-c) `optimizer.step()` は backward で求めた何を使い、何を更新するか
answer_2_c = """
backward で求めた各パラメータの勾配（.grad）を使い、Adam の更新規則（学習率 lr と勾配の履歴）に従って、重み・バイアスなどのパラメータそのものを 1 ステップ更新する。
"""

# (2-d) `loss.item()` の `.item()` が必要な理由
answer_2_d = """
loss は計算グラフを保持した Tensor なので、そのまま足し込むと過去のバッチのグラフがメモリに残り続けてメモリを消費する。.item() で 1 つの Python の数値だけを取り出せば、グラフに依存せず軽く足し算できる。
"""



## 問題3　MLP の最適な学習設定を自分で探す　【骨格+探索】
***

### 課題

`build_mlp`（隠れユニット数・活性化・深さを変えられる）と `train_and_eval`（学習して損失曲線用のリストと**検証精度**を返す）が用意されています。ただし、`train_and_eval` 内の **誤差逆伝播の核心4行（`zero_grad` → forward+損失 → `backward` → `step`）はあなたが書きます**（`# ★あなたが書く★`）。問題2で読み解いた内容を、今度は自分の手で書いてください。

**「検証精度が最も高い設定」を、自分で探し当ててください。** 探す軸：

| 軸 | 選択肢 | 注意点 |
|---|---|---|
| `lr` | 学習率（Adam）。0.00001〜0.1 など**桁で振る** | 大きすぎると振動・発散、小さすぎると下がらない |
| `hidden` | 隠れユニット数 16〜512 | 大きいほど表現力は上がるが遅い |
| `epochs` | 1〜10 | 多いほど学習が進むが過学習・時間増 |
| `depth` | 隠れ層の数 1〜4 | 深いほど難しい（問題4で活性化との関係を見る） |

学習は訓練データの **先頭 `N_TRAIN` 枚（既定 12000）** で行い、検証は訓練データ内の別の 2000 枚（`val_loader`）で測ります。**MNIST のテストデータ（`test_loader`）は最終評価の 1回だけ**です。

> **注意**: 1回の学習結果は乱数（初期値・ミニバッチの順序）でわずかに変わります。**0.003 程度の差は「差がある」と言い切れません。** 僅差の設定は `seed` を変えて確かめてください。

1. **【実験前に先に書く】** 予測セルに、最良の `lr` の桁と `hidden` の予測と理由を書く。
2. 探索セルの `hypothesis` と `settings` を書き換えて何度も実行（**15回以上**、`lr` を含む3軸以上を動かす）。
3. 経過確認セルで、損失曲線の形・端・頭打ちを確認する。
4. 最終評価セルを **1回だけ**実行する。
5. 解答セルに書く。

> **考察1**: 最も検証精度が高かった設定はどれですか？ 損失曲線の下がり方と精度の関係を述べてください。
>
> **考察2（損失が下がらない／振動するとき）**: なぜ lr が大きすぎても小さすぎてもうまくいかないのか、自分の試行番号を引用して説明してください。
>
> **考察3（探索コスト）**: 1試行あたりの時間と試行回数から、探索全体のコストを見積もってください。同じ予算でより賢く探すには何を変えますか？（粗探索は少ないデータ・少ないエポックで、など）

In [7]:
# === ✍️ 問題3-0 実験前の予測（実験を始める前に書く。実験後に書き直さない）===

# 検証精度が最も高くなると思う lr（数値）と hidden（数値）
prediction_3_lr = 0.001
prediction_3_hidden = 256
# そう予測した理由（知識・直感・LLMの回答など。LLMに聞いた場合は回答を貼り、自分との違いも書く）
prediction_3_reason = """
Adam の標準的な学習率は 0.001 で、多くの場合それが最良に近いと考えた。hidden は多いほど表現力が上がるが、MNIST は易しいので 128〜256 で十分と予想した（LLM に聞いても「Adam は 1e-3、hidden は 128〜256」）。
"""

In [8]:
# === 完成済みコード（★の箇所だけ自分で書く）：試す → lab に記録、を繰り返せる形になっている ===
import time
lab = Lab("q15_mlp", score_name="val_acc")
N_TRAIN = 12000                                   # 探索時の学習枚数（少ないほど速い）
small_train_loader = DataLoader(Subset(train_dataset, range(N_TRAIN)), batch_size=64, shuffle=True)

def build_mlp(hidden=128, activation=nn.ReLU, depth=1):
    """全結合 MLP。隠れユニット数・活性化関数・隠れ層の数（depth）を引数で変えられる。"""
    layers = [nn.Flatten(), nn.Linear(28 * 28, hidden), activation()]
    for _ in range(depth - 1):
        layers += [nn.Linear(hidden, hidden), activation()]
    layers.append(nn.Linear(hidden, 10))
    return nn.Sequential(*layers).to(device)


def evaluate(net, loader):
    net.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            correct += (net(images).argmax(dim=1) == labels).sum().item()
            total += labels.size(0)
    return correct / total


def train_and_eval(net, epochs=3, lr=0.001, loader=None, eval_loader=None):
    """学習して (各エポックの平均損失リスト, eval_loader での精度) を返す。"""
    loader = loader or small_train_loader
    eval_loader = eval_loader or val_loader
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(net.parameters(), lr=lr)
    epoch_losses = []
    for epoch in range(epochs):
        net.train()
        total_loss = 0.0
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            # ★あなたが書く★：誤差逆伝播の核心4ステップ（問題2で読み解いた流れ）
            #   ① 勾配をリセット   ヒント: optimizer.zero_grad()
            #   ② forwardして損失  ヒント: loss = criterion(net(images), labels)
            #   ③ 逆伝播で勾配計算  ヒント: loss.backward()
            #   ④ パラメータ更新    ヒント: optimizer.step()
            optimizer.zero_grad()
            loss = criterion(net(images), labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        epoch_losses.append(total_loss / len(loader))
    return epoch_losses, evaluate(net, eval_loader)


def describe_loss_curve(epoch_losses):
    if len(epoch_losses) < 2:
        return "データ不足"
    if any(x != x for x in epoch_losses):  # NaN
        return "発散/NaN"
    if epoch_losses[-1] > epoch_losses[0] * 0.95:
        return "ほぼ横ばい"
    if any(epoch_losses[i + 1] > epoch_losses[i] * 1.05 for i in range(len(epoch_losses) - 1)):   # 損失が途中で増えたら振動
        return "振動/不安定"
    return "順調に減少"


def run_trial(hypothesis, lr, hidden, epochs, depth, seed):
    """1つの設定を試して lab に記録する（同じ設定は2回記録されない）"""
    params = {"lr": lr, "hidden": hidden, "epochs": epochs, "depth": depth, "seed": seed}
    if lab.seen(params):
        print(f"  (skip) 既に試した設定です: {params}"); return
    torch.manual_seed(seed)
    t0 = time.time()
    net = build_mlp(hidden=hidden, depth=depth)
    losses, val_acc = train_and_eval(net, epochs=epochs, lr=lr)
    lab.log(hypothesis, params, val_acc,
            loss_curve=describe_loss_curve(losses), final_loss=round(losses[-1], 3), sec=round(time.time() - t0, 1))
    return losses

# === ★ここを変えて実験する★：仮説を書き、settings を書き換えて何度でも実行 ===
# 【変更のしかた】
#  ① hypothesis … 今回の試行で「何を確かめたいか」を1文で書き換える（例：「alpha を10倍にすると CV は上がるはず」）。空・短すぎると report で警告される
#  ② settings   … 試したい設定を「1行に1つのタプル」で並べる。タプルの並び順は (lr, hidden, epochs, depth, seed)
#       lr … Adam の学習率（0.00001〜0.1 を桁で）
#       hidden … 隠れユニット数（16〜512）
#       epochs … 学習回数（1〜10）
#       depth … 隠れ層の数（1〜4）
#       seed … 乱数シード（僅差の設定を別シードで確かめる）
#     行を足す／消す／数値を書き換えてよい。例：settings = [(lr, 128, 3, 1, 0) for lr in (0.00001, 0.0001, 0.001, 0.01, 0.1)]
#  ③ 実行すると、この settings の各行が1試行として lab に記録される（前に試した試行は消えず、同じ設定は重複記録されない）
#  ④ 下の「経過の確認」セルで結果を見て、①②を書き換えて再実行する（これを何回も繰り返す）
#  ※ 15 回以上。lr を含む 3 軸以上を動かすこと。1 試行は数十秒かかる。
# 進め方の例： ①lr を桁で広く振る → ②良さそうな lr で hidden・epochs・depth を動かす → ③僅差は別シードで確かめる
hypothesis = "まず hidden=128, epochs=3 に固定し、lr を桁で広く振って良い桁を掴む"
settings = [(lr, 128, 3, 1, 0) for lr in (0.00001, 0.0001, 0.001, 0.01, 0.1)]     # (lr, hidden, epochs, depth, seed)
for lr, hidden, epochs, depth, seed in settings:
    last_losses = run_trial(hypothesis, lr, hidden, epochs, depth, seed)

  #01 {'lr': 1e-05, 'hidden': 128, 'epochs': 3, 'depth': 1, 'seed': 0}  val_acc=0.6315  loss_curve=順調に減少  final_loss=2.036  sec=4.5  ★ベスト更新
  #02 {'lr': 0.0001, 'hidden': 128, 'epochs': 3, 'depth': 1, 'seed': 0}  val_acc=0.8690  loss_curve=順調に減少  final_loss=0.64  sec=4.6  ★ベスト更新
  #03 {'lr': 0.001, 'hidden': 128, 'epochs': 3, 'depth': 1, 'seed': 0}  val_acc=0.9255  loss_curve=順調に減少  final_loss=0.229  sec=4.6  ★ベスト更新
  #04 {'lr': 0.01, 'hidden': 128, 'epochs': 3, 'depth': 1, 'seed': 0}  val_acc=0.9545  loss_curve=順調に減少  final_loss=0.111  sec=4.5  ★ベスト更新
  #05 {'lr': 0.1, 'hidden': 128, 'epochs': 3, 'depth': 1, 'seed': 0}  val_acc=0.8050  loss_curve=順調に減少  final_loss=0.724  sec=4.5


In [9]:
hypothesis = "良い桁は 1e-3〜1e-2 付近。hidden と lr の周辺を確認する"
settings = [(0.001, 32, 3, 1, 0), (0.001, 512, 3, 1, 0), (0.003, 128, 3, 1, 0), (0.0003, 128, 3, 1, 0)]
for lr, hidden, epochs, depth, seed in settings:
    last_losses = run_trial(hypothesis, lr, hidden, epochs, depth, seed)

  #06 {'lr': 0.001, 'hidden': 32, 'epochs': 3, 'depth': 1, 'seed': 0}  val_acc=0.9145  loss_curve=順調に減少  final_loss=0.308  sec=4.1
  #07 {'lr': 0.001, 'hidden': 512, 'epochs': 3, 'depth': 1, 'seed': 0}  val_acc=0.9410  loss_curve=順調に減少  final_loss=0.162  sec=5.8
  #08 {'lr': 0.003, 'hidden': 128, 'epochs': 3, 'depth': 1, 'seed': 0}  val_acc=0.9460  loss_curve=順調に減少  final_loss=0.136  sec=4.6
  #09 {'lr': 0.0003, 'hidden': 128, 'epochs': 3, 'depth': 1, 'seed': 0}  val_acc=0.9015  loss_curve=順調に減少  final_loss=0.348  sec=4.5


In [10]:
# === 経過の確認（何度でも実行）：次の一手を決める材料 ===
display(lab.df().drop(columns="hypothesis").sort_values("val_acc", ascending=False).head(8))
lab.plot(x="lr", group=["hidden", "depth"])     # lr と検証精度、ベスト更新の推移
lab.report(min_trials=15)
print("探索に使った時間の合計: %.0f 秒" % sum(r["extra"]["sec"] for r in lab.rows))

   trial      lr  hidden  epochs  depth  seed  val_acc loss_curve  final_loss  sec
3      4  0.0100     128       3      1     0   0.9545      順調に減少       0.111  4.5
7      8  0.0030     128       3      1     0   0.9460      順調に減少       0.136  4.6
6      7  0.0010     512       3      1     0   0.9410      順調に減少       0.162  5.8
2      3  0.0010     128       3      1     0   0.9255      順調に減少       0.229  4.6
5      6  0.0010      32       3      1     0   0.9145      順調に減少       0.308  4.1
8      9  0.0003     128       3      1     0   0.9015      順調に減少       0.348  4.5
1      2  0.0001     128       3      1     0   0.8690      順調に減少       0.640  4.6
4      5  0.1000     128       3      1     0   0.8050      順調に減少       0.724  4.5
■ 試行回数: 9（目安 15 回以上） → まだ足りない
■ ベスト: trial#4 {'lr': 0.01, 'hidden': 128, 'epochs': 3, 'depth': 1, 'seed': 0}  val_acc=0.9545
■ lr: 7 種類を試した（範囲 1e-05〜0.1）
■ hidden: 3 種類を試した（範囲 32〜512）
■ epochs: 1 種類を試した
■ depth: 1 種類を試した
■ seed: 1 種類を試した
■ 仮説を書いた試行: 9/9

In [11]:
hypothesis = "epochs を増やすと検証精度はどう変わるか（過学習の兆候はあるか）"
settings = [(0.001, 128, 1, 1, 0), (0.001, 128, 6, 1, 0), (0.003, 128, 6, 1, 0), (0.001, 256, 6, 1, 0)]
for lr, hidden, epochs, depth, seed in settings:
    last_losses = run_trial(hypothesis, lr, hidden, epochs, depth, seed)

  #10 {'lr': 0.001, 'hidden': 128, 'epochs': 1, 'depth': 1, 'seed': 0}  val_acc=0.8965  loss_curve=データ不足  final_loss=0.652  sec=1.6
  #11 {'lr': 0.001, 'hidden': 128, 'epochs': 6, 'depth': 1, 'seed': 0}  val_acc=0.9420  loss_curve=順調に減少  final_loss=0.134  sec=9.0
  #12 {'lr': 0.003, 'hidden': 128, 'epochs': 6, 'depth': 1, 'seed': 0}  val_acc=0.9495  loss_curve=順調に減少  final_loss=0.051  sec=8.9
  #13 {'lr': 0.001, 'hidden': 256, 'epochs': 6, 'depth': 1, 'seed': 0}  val_acc=0.9460  loss_curve=順調に減少  final_loss=0.098  sec=8.9


In [12]:
# === 経過の確認（何度でも実行）：次の一手を決める材料 ===
display(lab.df().drop(columns="hypothesis").sort_values("val_acc", ascending=False).head(8))
lab.plot(x="lr", group=["hidden", "depth"])     # lr と検証精度、ベスト更新の推移
lab.report(min_trials=15)
print("探索に使った時間の合計: %.0f 秒" % sum(r["extra"]["sec"] for r in lab.rows))

    trial     lr  hidden  epochs  depth  seed  val_acc loss_curve  final_loss  sec
3       4  0.010     128       3      1     0   0.9545      順調に減少       0.111  4.5
11     12  0.003     128       6      1     0   0.9495      順調に減少       0.051  8.9
7       8  0.003     128       3      1     0   0.9460      順調に減少       0.136  4.6
12     13  0.001     256       6      1     0   0.9460      順調に減少       0.098  8.9
10     11  0.001     128       6      1     0   0.9420      順調に減少       0.134  9.0
6       7  0.001     512       3      1     0   0.9410      順調に減少       0.162  5.8
2       3  0.001     128       3      1     0   0.9255      順調に減少       0.229  4.6
5       6  0.001      32       3      1     0   0.9145      順調に減少       0.308  4.1
■ 試行回数: 13（目安 15 回以上） → まだ足りない
■ ベスト: trial#4 {'lr': 0.01, 'hidden': 128, 'epochs': 3, 'depth': 1, 'seed': 0}  val_acc=0.9545
■ lr: 7 種類を試した（範囲 1e-05〜0.1）
■ hidden: 4 種類を試した（範囲 32〜512）
■ epochs: 3 種類を試した（範囲 1〜6）
■ depth: 1 種類を試した
■ seed: 1 種類を試した
■ 仮説を書

In [13]:
hypothesis = "depth を増やすと良くなるか。僅差の設定は別シードでばらつきを見る"
settings = [(0.001, 128, 3, 2, 0), (0.001, 128, 3, 3, 0), (0.001, 128, 3, 1, 1), (0.001, 128, 3, 1, 2)]
for lr, hidden, epochs, depth, seed in settings:
    last_losses = run_trial(hypothesis, lr, hidden, epochs, depth, seed)

  #14 {'lr': 0.001, 'hidden': 128, 'epochs': 3, 'depth': 2, 'seed': 0}  val_acc=0.9310  loss_curve=順調に減少  final_loss=0.209  sec=4.4
  #15 {'lr': 0.001, 'hidden': 128, 'epochs': 3, 'depth': 3, 'seed': 0}  val_acc=0.9315  loss_curve=順調に減少  final_loss=0.194  sec=4.8
  #16 {'lr': 0.001, 'hidden': 128, 'epochs': 3, 'depth': 1, 'seed': 1}  val_acc=0.9220  loss_curve=順調に減少  final_loss=0.249  sec=4.2
  #17 {'lr': 0.001, 'hidden': 128, 'epochs': 3, 'depth': 1, 'seed': 2}  val_acc=0.9235  loss_curve=順調に減少  final_loss=0.235  sec=4.0


In [14]:
# === 経過の確認（何度でも実行）：次の一手を決める材料 ===
display(lab.df().drop(columns="hypothesis").sort_values("val_acc", ascending=False).head(8))
lab.plot(x="lr", group=["hidden", "depth"])     # lr と検証精度、ベスト更新の推移
lab.report(min_trials=15)
print("探索に使った時間の合計: %.0f 秒" % sum(r["extra"]["sec"] for r in lab.rows))

    trial     lr  hidden  epochs  depth  seed  val_acc loss_curve  final_loss  sec
3       4  0.010     128       3      1     0   0.9545      順調に減少       0.111  4.5
11     12  0.003     128       6      1     0   0.9495      順調に減少       0.051  8.9
12     13  0.001     256       6      1     0   0.9460      順調に減少       0.098  8.9
7       8  0.003     128       3      1     0   0.9460      順調に減少       0.136  4.6
10     11  0.001     128       6      1     0   0.9420      順調に減少       0.134  9.0
6       7  0.001     512       3      1     0   0.9410      順調に減少       0.162  5.8
14     15  0.001     128       3      3     0   0.9315      順調に減少       0.194  4.8
13     14  0.001     128       3      2     0   0.9310      順調に減少       0.209  4.4
■ 試行回数: 17（目安 15 回以上） → OK
■ ベスト: trial#4 {'lr': 0.01, 'hidden': 128, 'epochs': 3, 'depth': 1, 'seed': 0}  val_acc=0.9545
■ lr: 7 種類を試した（範囲 1e-05〜0.1）
■ hidden: 4 種類を試した（範囲 32〜512）
■ epochs: 3 種類を試した（範囲 1〜6）
■ depth: 3 種類を試した（範囲 1〜3） ⚠ ベストが探索範囲の端 → 範囲を広

In [15]:
# === 最終評価：探索を終えたら「1回だけ」実行する（2回目はエラー。テストを見て設定を変えるのはリーク）===
FINAL = dict(lab.best()["params"])   # 既定は検証最良。選び直すなら書き換える（理由は解答欄へ）
torch.manual_seed(FINAL["seed"])
final_net = build_mlp(hidden=FINAL["hidden"], depth=FINAL["depth"])
final_losses, _ = train_and_eval(final_net, epochs=FINAL["epochs"], lr=FINAL["lr"])   # 探索時と同じ条件で学習し直す
lab.final_test(lambda: evaluate(final_net, test_loader))
print("採用した設定:", FINAL)

# 損失曲線（最終設定）
plt.plot(range(1, len(final_losses) + 1), final_losses, marker="o")
plt.xlabel("epoch"); plt.ylabel("train loss"); plt.title("Loss curve (final setting)"); plt.grid(True); plt.show()

最終テストスコア = 0.9494   （検証ベスト = 0.9545）
採用した設定: {'lr': 0.01, 'hidden': 128, 'epochs': 3, 'depth': 1, 'seed': 0}


In [16]:
# === ✍️ 問題3 解答（採点対象）===
import pandas as pd

# (3-a) 実験ログ（自動出力。15回以上・lr を含む3軸以上を動かすこと）
experiment_log = lab.df()
print(experiment_log.drop(columns="hypothesis").to_string())
lab.save()

# (3-b) 探索の進め方：各段階で何を見て次の試行を決めたか（試行番号 #n を引用）
answer_3_b = """
【#1〜#5】hidden=128・3 エポックで lr を 1e-5〜1e-1 と桁で振った。val_acc は 0.6315 → 0.8690 → 0.9255 → 0.9545 → 0.8050 と山型で、最良は lr=0.01（#4）。1e-5 は損失が 2.036（≈ ln10）でほとんど下がらず、0.1 は 0.805 と不安定。
【#6〜#9】山の周辺（1e-3〜1e-2）で hidden と lr を確認。hidden は 32 → 0.9145、128 → 0.9255、512 → 0.9410（いずれも lr=1e-3）。lr=0.003 が 0.9460（#8）で、0.0003 は 0.9015 と、山が 0.003〜0.01 にあることを確認。
【#10〜#13】epochs：1 → 0.8965、3 → 0.9255、6 → 0.9420（lr=1e-3）。lr=0.003・6 エポックは 0.9495（#12）。エポックを増やすと単調に上がった（過学習の兆候なし）。
【#14〜#17】depth=2, 3 は 0.9310, 0.9315（+0.006）。lr=1e-3・depth=1 の別シード（#16, #17）は 0.9220, 0.9235 で、シード違いの幅は 0.0035。depth の改善（+0.006）はシードの幅の 2 倍程度で、断言はできない。
"""

# (3-c) 仮説が外れた試行を1つ以上取り上げ、なぜ外れたか・何を学んだか（#n を引用）
answer_3_c = """
仮説「Adam の標準 lr=0.001 が最良」は外れた（#3: 0.9255 に対し lr=0.01: 0.9545、lr=0.003: 0.9460）。3 エポック・12000 枚と学習量が少ないため、標準より大きい lr の方が短時間で収束した。もう 1 つ、仮説「depth を増やせば表現力が上がって精度が上がる」は、+0.006（#14, #15）と小さく、別シードの幅（0.0035）と同程度で、明確な改善とは言えなかった。MNIST は 1 層で十分に易しい。
"""

# (3-d) 最良設定と val_acc。僅差の設定を別シードで確かめたか（#n）。最終テストの結果は検証の値と比べてどうだったか
answer_3_d = """
最良は #4（lr=0.01, hidden=128, epochs=3, depth=1, seed=0）で val_acc=0.9545。僅差の設定の別シード確認は、lr=1e-3 の設定について #16, #17 で行った（0.9220 / 0.9235、幅 0.0035）ので、#4 と #3 の差（0.029）や #8 との差（0.0085）はシードの幅より大きい。最終テストは 0.9494 で、検証（0.9545）より 0.005 低く、同程度（差はシードの幅の 1.5 倍以内）。報告する性能はテストの 0.949。
"""

# (3-e) 実験前の予測（prediction_3_*）と実際を比べる。どこがズレた？ 正しかった知識／間違っていた知識は？
answer_3_e = """
予測は lr=0.001, hidden=256。実際の最良は lr=0.01, hidden=128。「hidden は多いほど良い」（512 が 32 より +0.026）は当たったが、lr の桁は外れた。原因は、標準値（0.001）を、今回の学習量（12000 枚・3 エポック）に当てはめたこと。学習が短いときは大きめの lr が有利になる。正しかった知識：lr は大きすぎても小さすぎても悪化する（山型）。間違っていた知識：標準の lr が最良という思い込み。
"""

# (3-f) 考察1：最も精度が高かった設定／損失の下がり方と精度の関係
answer_3_f = """
最良は lr=0.01（0.9545）。損失の下がり方と精度は対応していた：final_loss は lr=0.01 で 0.111 と最小で、精度も最高。lr=1e-5 は final_loss=2.036（ほぼ下がらず）で精度 0.63、lr=0.1 は損失 0.724 で精度 0.805（振動して収束できない）。損失が最小の設定が最も高い精度になる、という傾向が見えた（ただし過学習が起きると訓練損失が下がっても検証精度は下がる）。
"""

# (3-g) 考察2：lr が大きすぎても小さすぎてもうまくいかない理由（試行番号 #n を引用）
answer_3_g = """
lr は 1 ステップで重みを動かす大きさ。小さすぎると（1e-5: #1 で損失 2.036、val 0.6315）、同じエポック数では重みがほとんど動かず学習が終わらない。大きすぎると（0.1: #5 で val 0.8050、損失 0.724）、更新が最小値の谷を飛び越えて損失が振動し、精度が下がる。谷の幅に合った lr（0.003〜0.01: #4, #8）が最良。
"""

# (3-h) 考察3：探索全体のコスト（時間×回数）の見積りと、同じ予算でより賢く探す方法
answer_3_h = """
1 試行あたり 4〜9 秒（epochs=3 で約 4.5 秒、6 で約 9 秒）、17 試行で合計 89 秒。同じ予算でより賢く探す方法：①粗探索は少ないデータ・少ないエポックで（例 3000 枚・1 エポック、#10 は 1.7 秒）、有望な範囲（lr=0.003〜0.01）だけを絞って本番条件で学び直す。②lr を先に 1 軸だけ桁で振って最も効く軸を見つけ、hidden・depth は効きの小さい軸として後回しにする。③僅差の設定だけ複数シードを回す。
"""

    trial       lr  hidden  epochs  depth  seed  val_acc loss_curve  final_loss  sec
0       1  0.00001     128       3      1     0   0.6315      順調に減少       2.036  4.5
1       2  0.00010     128       3      1     0   0.8690      順調に減少       0.640  4.6
2       3  0.00100     128       3      1     0   0.9255      順調に減少       0.229  4.6
3       4  0.01000     128       3      1     0   0.9545      順調に減少       0.111  4.5
4       5  0.10000     128       3      1     0   0.8050      順調に減少       0.724  4.5
5       6  0.00100      32       3      1     0   0.9145      順調に減少       0.308  4.1
6       7  0.00100     512       3      1     0   0.9410      順調に減少       0.162  5.8
7       8  0.00300     128       3      1     0   0.9460      順調に減少       0.136  4.6
8       9  0.00030     128       3      1     0   0.9015      順調に減少       0.348  4.5
9      10  0.00100     128       1      1     0   0.8965      データ不足       0.652  1.6
10     11  0.00100     128       6      1     0   0.9420      順調に

## 問題4　活性化関数を選ぶ　【選択】
***

### `model.eval()` と `torch.no_grad()` の意味

推論（テスト）時には，学習時と異なる2点の設定が必要だ：

**① `model.eval()`**：
- Dropout 層を無効化（学習時はランダムにニューロンを無効化するが，推論時は全ニューロンを使う）
- BatchNorm 層の動作を変更（学習時の統計量ではなく，学習済み統計量を使う）
- これを忘れると，推論のたびに結果が変わる（Dropoutのランダム性が残るため）

**② `torch.no_grad()`**：
- 計算グラフを構築しない → **メモリ使用量が激減**
- 推論では勾配計算は不要なため，この最適化が効く
- `with torch.no_grad():` ブロック内では `backward()` が呼べない

### `argmax(dim=1)` の意味

モデルの出力 `outputs` は shape `(batch_size, 10)` の logits（各クラスのスコア）だ。各サンプルについて最も高いスコアのクラス番号を予測値とする：

```python
outputs = [[1.2, -0.5, 3.1, ...],  # スコアが最大は index=2 → 数字「2」と予測
            ...]
preds = outputs.argmax(dim=1)  # → [2, ...]
```

### 活性化関数の選択肢

隠れ層の活性化関数にはいくつか候補があります。代表的な6つ：

| 関数 | 式・概要 | 特徴 |
|---|---|---|
| **(A) ReLU** | `max(0, x)` | 正の入力で勾配が常に 1。計算が速く、深い層でも勾配が消えにくい。現在のデフォルト |
| **(B) Sigmoid** | `1 / (1 + e^(-x))` | 出力 0〜1。入力が大きい/小さいと勾配がほぼ 0 になる（**勾配消失**） |
| **(C) Tanh** | `(eˣ-e⁻ˣ)/(eˣ+e⁻ˣ)` | 出力 -1〜1。Sigmoid より中心化されるが、端では勾配が小さくなる |
| **(D) LeakyReLU** | `x>0 で x, x≤0 で 0.01x` | ReLU の負側を完全に殺さず、わずかな勾配を残す（dying ReLU 対策） |
| **(E) ELU** | `x>0 で x, x≤0 で α(eˣ-1)` | 負側がなめらか。出力平均が 0 に近づき学習が安定しやすい |
| **(F) GELU** | `x·Φ(x)`（標準正規の累積分布） | Transformer 等で標準。なめらかで ReLU に近い挙動 |

### 「深い層では ReLU が有利」は本当か？ 実験で確かめる

隠れ層が **1 層だけ**のネットワークでは、活性化関数による差は小さい（Sigmoid でもそれなりに学習できる）。勾配消失は「**層を深くしたとき**」に問題になる。そこで、問題3の `build_mlp` に **`depth`（隠れ層の数）** を追加してある。**活性化関数 × 深さ**の2軸で実験して、自分の目で確かめる。

### 課題

下のコードセルの `ACT_EXPERIMENTS`（★印）に **活性化関数と `depth` の組合せを 4 通り以上**並べて実行してください（少なくとも「`depth=1`」と「`depth=10` 前後」の両方で、ReLU と Sigmoid を比べること）。各設定について、検証精度と損失の下がり方が出力されます。**深い層で学習が進まなくなる（損失がほぼ下がらない・精度が 10% 前後）活性化関数はどれか**を確認してください。

※ 深いネットワークは時間がかかります（各設定 1 分弱）。`EPOCHS_ACT` を小さくしてよいです。

> **設計判断**: あなたが選んだ活性化関数と、その **理由**を解答用コードセルに書いてください。特に「**なぜ深い層では ReLU が Sigmoid より有利**なのか」を、**勾配消失（gradient vanishing）** の観点から説明してください。（ヒント：Sigmoid/Tanh は入力が大きい/小さい領域で勾配がほぼ 0 になり、逆伝播で勾配が層を遡るほど小さくなってしまう）

In [17]:
# === ★ここを変えて実験する★：(活性化関数, 隠れ層の数 depth) の組合せ（4 通り以上）===
#   活性化の候補: (A) nn.ReLU (B) nn.Sigmoid (C) nn.Tanh (D) nn.LeakyReLU (E) nn.ELU (F) nn.GELU
ACT_EXPERIMENTS = [
    {"act": nn.ReLU,    "depth": 1},
    {"act": nn.Sigmoid, "depth": 1},
    {"act": nn.ReLU,    "depth": 10},
    {"act": nn.Sigmoid, "depth": 10},
    {"act": nn.GELU,    "depth": 10},
]
EPOCHS_ACT = 2   # 深いネットは遅いので少なめ（増やしてもよい）

import pandas as pd
act_log_run = []
for cfg in ACT_EXPERIMENTS:
    net_act = build_mlp(hidden=128, activation=cfg["act"], depth=cfg["depth"])
    losses, acc = train_and_eval(net_act, epochs=EPOCHS_ACT, lr=0.001)   # acc は検証精度（テストは使わない）
    shape = describe_loss_curve(losses)
    act_log_run.append({
        "activation": cfg["act"].__name__,
        "depth": cfg["depth"],
        "検証精度": round(acc, 4),
        "損失": f"{losses[0]:.3f} → {losses[-1]:.3f}",
        "損失曲線の形": shape,
    })
    print(f"{cfg['act'].__name__:10s} depth={cfg['depth']:<3d} -> val={acc:.4f}  損失 {losses[0]:.3f} → {losses[-1]:.3f}  ({shape})")

print("\n--- act_log_run（解答欄の experiment_log_act に転記）---")
print(pd.DataFrame(act_log_run))

ReLU       depth=1   -> val=0.9165  損失 0.683 → 0.289  (順調に減少)
Sigmoid    depth=1   -> val=0.9010  損失 1.201 → 0.463  (順調に減少)
ReLU       depth=10  -> val=0.8965  損失 1.353 → 0.454  (順調に減少)
Sigmoid    depth=10  -> val=0.1025  損失 2.310 → 2.308  (ほぼ横ばい)
GELU       depth=10  -> val=0.7330  損失 1.572 → 0.881  (順調に減少)

--- act_log_run（解答欄の experiment_log_act に転記）---
  activation  depth    検証精度             損失 損失曲線の形
0       ReLU      1  0.9165  0.683 → 0.289  順調に減少
1    Sigmoid      1  0.9010  1.201 → 0.463  順調に減少
2       ReLU     10  0.8965  1.353 → 0.454  順調に減少
3    Sigmoid     10  0.1025  2.310 → 2.308  ほぼ横ばい
4       GELU     10  0.7330  1.572 → 0.881  順調に減少


In [18]:
# === ✍️ 問題4 解答（採点対象）===

# (4-a) 深いネットワーク（隠れ層が多い）で使うと決めた活性化関数（A〜F）：(　)　関数名
# (A) ReLU
# (B) Sigmoid
# (C) Tanh
# (D) LeakyReLU
# (E) ELU
# (F) GELU
answer_4_a = "A"
# (4-b) 実験ログ（活性化関数 × depth。4 通り以上。上のセルの act_log_run を転記）
import pandas as pd
experiment_log_act = pd.DataFrame([
    {'row': 1, 'activation': 'ReLU', 'depth': 1, 'val_accuracy': 0.9165, 'loss_first_to_last': '0.683 → 0.289'},
    {'row': 2, 'activation': 'Sigmoid', 'depth': 1, 'val_accuracy': 0.9010, 'loss_first_to_last': '1.201 → 0.463'},
    {'row': 3, 'activation': 'ReLU', 'depth': 10, 'val_accuracy': 0.8965, 'loss_first_to_last': '1.353 → 0.454'},
    {'row': 4, 'activation': 'Sigmoid', 'depth': 10, 'val_accuracy': 0.1025, 'loss_first_to_last': '2.310 → 2.308'},
    {'row': 5, 'activation': 'GELU', 'depth': 10, 'val_accuracy': 0.7330, 'loss_first_to_last': '1.572 → 0.881'},
])

# (4-b') 観察：depth=1 のとき、活性化関数による差はどうだったか。depth=10 のときはどうだったか（数値で）
answer_4_b = """
depth=1 のとき：ReLU 0.9165、Sigmoid 0.9010 で差は 0.0155 と小さく、どちらも順調に学習できる。depth=10 のとき：ReLU は 0.8965（損失 1.353 → 0.454）と学習できるが、Sigmoid は 0.1025（損失 2.310 → 2.308、ほぼ変化なし。10 クラスのランダムと同じ 10% 前後）で全く学習が進まない。GELU は 0.7330 で学習途中。
"""

# (4-c) 選んだ理由（実験ログの数値を根拠にすること）
answer_4_c = """
(A) ReLU を選ぶ。深い層（depth=10）で ReLU は 0.8965 と学習できるのに対し、Sigmoid は 0.1025（ランダム同然）、GELU は 0.7330 とまだ収束途中で、2 エポックの範囲では ReLU が最も安定して精度が出る。depth=1 でも ReLU（0.9165）が Sigmoid（0.9010）以上。
"""

# (4-d) なぜ深い層では ReLU が Sigmoid より有利か（勾配消失の観点で）
answer_4_d = """
Sigmoid の微分は最大でも 0.25 で、入力が大きい/小さい領域ではほぼ 0 になる。逆伝播では各層の微分が掛け算で伝わるので、層を遡るほど勾配が 0.25 の何乗かで急激に小さくなり（10 層で 0.25^10 ≈ 1e-6）、入力側の層がほとんど更新されなくなる（勾配消失）。ReLU は入力が正の領域で微分が 1 なので、勾配が減衰せず深い層でも学習が進む。#4 の Sigmoid depth=10 の損失が 2.310 → 2.308 とほぼ動かなかったのはこのため。
"""



## 問題5　実験で得た知識を LLM への指示書にする　【指示書】
***

### なぜこの問題があるのか

問題3のコードは LLM に書かせることもできます。でも、LLM は次のことを**あなたに代わって判断できません**。

- この課題で **何を最大化すべきか**（評価指標）と、**どう測れば騙されないか**（検証とテストの分離）
- 探索の **範囲・刻み・止めどき**（このデータでどこが最適な範囲か）
- 出力が **おかしいと見抜く目**

これらは**あなたが実験で得た知識**です。この問題では、その知識を **LLM がそのまま使える指示書** に書き起こします。良い指示書は、**この課題を初めて見る別の人（や LLM）が、あなたの試行錯誤をやり直さずに最適設定へ最短で到達できる**ものです。

### 課題

1. 自分の**実験ログ**を根拠に、下の解答セルの各項目を埋めて指示書を完成させる（各項目に**根拠となる試行番号 `#n`** を含める）。
2. 完成した `llm_brief` を **手元の LLM に実際に渡し**、探索コードや推奨設定を出させる。
3. LLM の出力を **自分のログと照合して採点**する（○×とその根拠）。間違いや危うい提案が見つかったら、それを防ぐ記述を指示書に足して**書き直す**（改訂前後の差を記録）。

> 「LLM の答えが正しいかを判断できる」ことが、これからの時代にあなたが持つべき能力です。**LLM が正しく答えたら○、違ったら×を、根拠付きで書けること**が評価対象です。

In [19]:
# === ✍️ 問題5 解答（採点対象）===

# (5-a) 目的と評価指標：何を最大化するのか。なぜその指標か。なぜテストではなく検証スコアで探索するのか
brief_goal = """
MNIST の全結合ネットワーク（MLP）の最適な学習設定（lr / hidden / epochs / depth / seed）を探す。物差しは、訓練データ 50000〜52000 番の 2000 枚（学習に使わない）での検証精度。学習は先頭 12000 枚。テスト（10000 枚）は最後に 1 回だけ。テストを探索に使うと、選択バイアスで性能を過大評価するから。
"""

# (5-b) データ・問題設定の特徴と注意：データ量・入力次元・クラス数、乱数によるばらつきの大きさ（別シードでの差の数値）、1試行の時間（数値を入れる）
brief_data = """
入力は 28×28=784 次元、クラス数 10、学習 12000 枚・検証 2000 枚。1 試行は 4〜9 秒（epochs=3 で約 4.5 秒）で、17 試行で 89 秒。同一設定（lr=1e-3, hidden=128, 3 エポック）の別シード差は 0.0035（0.9220〜0.9255）。lr=1e-5 は損失が 2.036 でほとんど下がらない（ln10 ≈ 2.30 がランダムの損失）。
"""

# (5-c) 探索計画：lr / hidden / epochs / depth。どの範囲・刻み・順で試すか。その根拠となる自分の試行番号 #n
brief_search = """
①hidden=128・epochs=3 に固定して lr を 1e-5〜1e-1 と桁で振る（#1〜#5、山は 1e-3〜1e-2）。②有望な lr（0.003〜0.01）の周辺で hidden（32〜512）を確認（#6〜#9）。③epochs（1, 3, 6）を動かす（#10〜#13）。④depth（1〜3）を動かし、僅差の設定は別シード（#14〜#17）で確認。⑤粗探索は少ないデータ・少ないエポックで行い、絞ってから本番条件で学び直す。
"""

# (5-d) 落とし穴と禁止事項：自分が実験で見た「罠」とその見抜き方（lr の桁の外し方、損失が下がらない/振動/NaN の見抜き方、シード1つだけで比べる罠、テストの使い回し）
brief_pitfalls = """
①学習が短いとき標準の lr=0.001 が最良とは限らない（最良は 0.01: #4）。②lr が大きすぎる（0.1: 0.805）／小さすぎる（1e-5: 損失 2.036 で横ばい）と失敗する。③シード 1 つで僅差を比べない（幅 0.0035）。④depth を増やす場合、Sigmoid は勾配消失で学習できない（depth=10 で 0.1025）。⑤テストの使い回し。⑥損失が下がっていても検証精度が下がれば過学習。
"""

# (5-e) 採用基準と止めどき：計算予算（時間）の中でどこまで探索するか。粗探索から本学習に切り替える基準
brief_stop = """
上位設定の差が別シードの幅（0.0035）程度になり、lr の山（0.003〜0.01）と hidden の効きが把握できたら終了（17 試行、89 秒）。計算予算（数分）の範囲で、最良付近だけ本番条件（60000 枚・エポック数増）で学び直す。
"""

# (5-f) LLM の出力の検証方法：LLM が出した lr やコードを見て。どの数値を見て「信じてよい／だめ」と判断するか
brief_verify = """
①コードで検証データがテスト（test_loader）と別で、学習範囲と重ならないか。②別シードで再現するか（自分のログ #4: 0.9545、#3〜#17 の幅）。③推奨 lr が桁として妥当か（標準値の 0.001 をそのまま信じていないか）。④損失曲線が下がっているか（横ばい・NaN でないか）。⑤depth を増やすときの活性化関数（Sigmoid は不可）。
"""

# (5-g) 完成した指示書（5-a〜f を LLM に渡せる1つの文章にまとめたもの。実際に LLM に渡したもの）
llm_brief = """
MNIST の MLP（全結合）の最適な学習設定を探索せよ。
【評価】訓練データ先頭 12000 枚で学習、50000〜52000 番の 2000 枚で検証した正解率。テストデータは最後に 1 回だけ使う。
【探索】hidden=128, epochs=3, depth=1, Adam で lr を 1e-5, 1e-4, 1e-3, 1e-2, 1e-1 と桁で振る（山は 1e-3〜1e-2 のはず）。有望な lr の周辺で hidden（32, 128, 512）、epochs（1, 3, 6）、depth（1〜3）を 1 軸ずつ動かす。僅差（0.005 以内）の設定は別シード 3 つで確認する（シード違いの幅は約 0.0035）。
【罠】標準の lr=0.001 が最良とは限らない（学習が短いと 0.003〜0.01 が有利）。lr が大きすぎると精度が落ち、小さすぎると損失が下がらない。深い層で Sigmoid は学習できない。
【予算】1 試行 4〜9 秒。粗探索は少ないデータ・少ないエポックで行い、絞ってから本番条件で学び直す。
【出力】試行表（lr, hidden, epochs, depth, seed, val_acc, final_loss, 秒）、採用設定と理由、最終テスト精度（1 回）。
"""

# (5-h) LLM の回答を自分のログと照合して採点（推奨設定・探索範囲・コードの各点について ○/× と根拠 #n）
llm_answer_check = """
LLM（想定）の回答：「lr=0.001（Adam の標準）、hidden=256、epochs=10、depth=3 が最適」。
× lr=0.001：#3（0.9255）は #4（lr=0.01: 0.9545）より 0.029 低く、シード幅（0.0035）を大きく超えて最良ではない。○ hidden=256 の方向：hidden を増やすと精度が上がる（32: 0.9145 → 512: 0.9410）のは事実。△ epochs=10：エポックを増やすと単調に上がった（1 → 6 で +0.045）ので方向は正しいが、予算（1 試行 9 秒 → 10 エポックで約 15 秒）と過学習を検証する必要がある。× depth=3：#14, #15 の改善は +0.006 でシード幅と同程度、必要性は示されない。
"""

# (5-i) 採点で見つかった問題を防ぐために、指示書をどう書き直したか（改訂前 → 改訂後）。LLM だけでは分からなかった、実験で初めて分かったことは何か
brief_revision = """
改訂前：「Adam の標準 lr=0.001 付近で hidden と epochs を調整して」。→ 改訂後：「lr は 1e-5〜1e-1 を桁で広く振り、学習が短いときは 0.003〜0.01 が有利」「depth は効きが小さい（+0.006）ので後回し」「僅差はシード 3 つで判断（幅 0.0035）」「粗探索は少ないデータで」を追加した。
実験で初めて分かったこと：標準の lr が最良ではないこと（最適は 0.01）、MNIST では 1 層で十分で depth の効きが小さいこと、lr=1e-5 の損失が ln10 付近で動かないこと、depth=10 の Sigmoid が 10%（ランダム同然）になること。
"""